# Análise de População — Censo 2022 vs. Censo 2010 (Compatibilizado)



## 1. Importar bibliotecas

In [1]:
import pandas as pd

pd.set_option('display.float_format', lambda x: f'{x:,.0f}')


## 2. Ler a tabela

A planilha `CD2022_Populacao_2010_Compatibilizada_20231222.xlsx` tem cabeçalho na linha 3 (índice 2), dados a partir da linha 4, e um rodapé com notas ao final que precisa ser descartado.

In [2]:
ARQUIVO = 'CD2022_Populacao_2010_Compatibilizada_20231222.xlsx'

df = pd.read_excel(
    ARQUIVO,
    sheet_name='Municípios',
    header=2,        
    usecols='B:H',     
)

df = df.dropna(subset=['COD. UF'])
df['COD. UF'] = df['COD. UF'].astype(int)

df.columns = [
    'UF', 'COD_UF', 'COD_MUNIC', 'MUNICIPIO',
    'POP_2010_SINOPSE', 'POP_2010_COMPATIBILIZADA', 'POP_2022'
]

print(df.shape)
df.head()


(5570, 7)


,UF,COD_UF,COD_MUNIC,MUNICIPIO,POP_2010_SINOPSE,POP_2010_COMPATIBILIZADA,POP_2022
0,RO,11,15,Alta Floresta D'Oeste,"24,392","24,392","21,494"
1,RO,11,23,Ariquemes,"90,353","90,353","96,833"
2,RO,11,31,Cabixi,"6,313","6,313","5,351"
3,RO,11,49,Cacoal,"78,574","78,574","86,887"
4,RO,11,56,Cerejeiras,"17,029","17,029","15,890"


## 3. Tabela agregada por Estado

Somamos a população 2010 (compatibilizada, que já leva em conta mudanças de limites territoriais) e a população 2022 de todos os municípios de cada UF.

In [3]:
pop_estado = (
    df.groupby('UF', as_index=False)
      .agg(
          POP_2010=('POP_2010_COMPATIBILIZADA', 'sum'),
          POP_2022=('POP_2022', 'sum'),
      )
)

pop_estado


,UF,POP_2010,POP_2022
0,AC,"733,559","830,018"
1,AL,"3,120,887","3,127,683"
2,AM,"3,483,985","3,941,613"
3,AP,"669,526","733,759"
4,BA,"14,017,071","14,141,626"
5,CE,"8,451,644","8,794,957"
6,DF,"2,572,159","2,817,381"
7,ES,"3,514,952","3,833,712"
8,GO,"6,001,789","7,056,495"
9,MA,"6,574,789","6,776,699"


## 4. Calcular crescimento (2022 - 2010) e ordenar

Calculamos a diferença absoluta e o crescimento percentual, e ordenamos do estado que mais cresceu para o que menos cresceu (ou mais encolheu).

In [4]:
pop_estado['CRESCIMENTO_ABS'] = pop_estado['POP_2022'] - pop_estado['POP_2010']
pop_estado['CRESCIMENTO_PCT'] = (pop_estado['CRESCIMENTO_ABS'] / pop_estado['POP_2010']) * 100

pop_estado = pop_estado.sort_values('CRESCIMENTO_ABS', ascending=False).reset_index(drop=True)

pop_estado


,UF,POP_2010,POP_2022,CRESCIMENTO_ABS,CRESCIMENTO_PCT
0,SP,"41,262,199","44,411,238","3,149,039",8
1,SC,"6,248,436","7,610,361","1,361,925",22
2,GO,"6,001,789","7,056,495","1,054,706",18
3,PR,"10,444,526","11,444,380","999,854",10
4,MG,"19,597,330","20,539,989","942,659",5
5,MT,"3,035,122","3,658,649","623,527",21
6,PA,"7,581,051","8,120,131","539,080",7
7,AM,"3,483,985","3,941,613","457,628",13
8,CE,"8,451,644","8,794,957","343,313",4
9,ES,"3,514,952","3,833,712","318,760",9


## 5. Salvar tabela por Estado em CSV

In [5]:
pop_estado.to_csv('populacao_por_estado_crescimento.csv', index=False, encoding='utf-8-sig')
print('Arquivo salvo: populacao_por_estado_crescimento.csv')


Arquivo salvo: populacao_por_estado_crescimento.csv


## 6. Trabalhar por Município

Agora repetimos a análise no nível de município, calculando a diferença de população entre 2010 (compatibilizada) e 2022, e ordenando do município que mais cresceu para o que mais encolheu.

In [6]:
pop_municipio = df[[
    'UF', 'COD_MUNIC', 'MUNICIPIO', 'POP_2010_COMPATIBILIZADA', 'POP_2022'
]].copy()

pop_municipio = pop_municipio.rename(columns={
    'POP_2010_COMPATIBILIZADA': 'POP_2010'
})

pop_municipio['CRESCIMENTO_ABS'] = pop_municipio['POP_2022'] - pop_municipio['POP_2010']
pop_municipio['CRESCIMENTO_PCT'] = (pop_municipio['CRESCIMENTO_ABS'] / pop_municipio['POP_2010']) * 100

pop_municipio = pop_municipio.sort_values('CRESCIMENTO_ABS', ascending=False).reset_index(drop=True)

pop_municipio.head(20)


,UF,COD_MUNIC,MUNICIPIO,POP_2010,POP_2022,CRESCIMENTO_ABS,CRESCIMENTO_PCT
0,AM,"2,603",Manaus,"1,802,014","2,063,689","261,675",15
1,DF,108,Brasília,"2,572,159","2,817,381","245,222",10
2,SP,"50,308",São Paulo,"11,253,503","11,451,999","198,496",2
3,SP,"52,205",Sorocaba,"586,816","723,682","136,866",23
4,GO,"8,707",Goiânia,"1,301,912","1,437,366","135,454",10
5,RR,100,Boa Vista,"284,313","413,486","129,173",45
6,SC,"5,407",Florianópolis,"421,240","537,211","115,971",28
7,PA,"5,536",Parauapebas,"153,908","267,836","113,928",74
8,MS,"2,704",Campo Grande,"786,774","898,100","111,326",14
9,PB,"7,507",João Pessoa,"723,515","833,932","110,417",15


## 7. Salvar tabela por Município em CSV

In [7]:
pop_municipio.to_csv('populacao_por_municipio_crescimento.csv', index=False, encoding='utf-8-sig')
print('Arquivo salvo: populacao_por_municipio_crescimento.csv')


Arquivo salvo: populacao_por_municipio_crescimento.csv


## 8. Conferência rápida

Os 5 municípios que mais cresceram e os 5 que mais encolheram em números absolutos.

In [8]:
print('Top 5 - maior crescimento:')
display(pop_municipio.head(5))

print('Top 5 - maior queda:')
display(pop_municipio.tail(5))


Top 5 - maior crescimento:


,UF,COD_MUNIC,MUNICIPIO,POP_2010,POP_2022,CRESCIMENTO_ABS,CRESCIMENTO_PCT
0,AM,"2,603",Manaus,"1,802,014","2,063,689","261,675",15
1,DF,108,Brasília,"2,572,159","2,817,381","245,222",10
2,SP,"50,308",São Paulo,"11,253,503","11,451,999","198,496",2
3,SP,"52,205",Sorocaba,"586,816","723,682","136,866",23
4,GO,"8,707",Goiânia,"1,301,912","1,437,366","135,454",10


Top 5 - maior queda:


,UF,COD_MUNIC,MUNICIPIO,POP_2010,POP_2022,CRESCIMENTO_ABS,CRESCIMENTO_PCT
5565,RS,"14,902",Porto Alegre,"1,409,351","1,332,845","-76,506",-5
5566,PA,"1,402",Belém,"1,393,399","1,303,403","-89,996",-6
5567,RJ,"4,904",São Gonçalo,"999,728","896,744","-102,984",-10
5568,RJ,"4,557",Rio de Janeiro,"6,320,446","6,211,223","-109,223",-2
5569,BA,"27,408",Salvador,"2,675,656","2,417,678","-257,978",-10
